In [ ]:
# week2_anomaly_detection.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

# -----------------------------
# 1. Load dataset
# -----------------------------
# Replace 'supply_chain_data.csv' with your actual file
df = pd.read_csv('supply_chain_data.csv')

# Ensure datetime column is parsed correctly
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Example: assume we have a 'sales' column
# If you have 'inventory' instead, adjust accordingly
series = df['sales']

# -----------------------------
# 2. Z-Score Method
# -----------------------------
df['z_score'] = (series - series.mean()) / series.std()
anomalies_z = df[np.abs(df['z_score']) > 3]

# -----------------------------
# 3. IQR Method
# -----------------------------
Q1 = series.quantile(0.25)
Q3 = series.quantile(0.75)
IQR = Q3 - Q1

df['iqr_flag'] = ((series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR))
anomalies_iqr = df[df['iqr_flag']]

# -----------------------------
# 4. Isolation Forest
# -----------------------------
iso_model = IsolationForest(contamination=0.05, random_state=42)
df['if_flag'] = iso_model.fit_predict(series.values.reshape(-1, 1))
anomalies_if = df[df['if_flag'] == -1]

# -----------------------------
# 5. Visualization
# -----------------------------
plt.figure(figsize=(12,6))
plt.plot(series.index, series.values, label='Sales', color='blue')

# Highlight anomalies from each method
plt.scatter(anomalies_z.index, anomalies_z['sales'], color='red', label='Z-Score Anomalies')
plt.scatter(anomalies_iqr.index, anomalies_iqr['sales'], color='green', label='IQR Anomalies')
plt.scatter(anomalies_if.index, anomalies_if['sales'], color='orange', label='Isolation Forest Anomalies')

plt.title('Sales Data with Anomaly Detection')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.show()

# -----------------------------
# 6. Save anomalies for reporting
# -----------------------------
anomalies_z.to_csv('anomalies_zscore.csv')
anomalies_iqr.to_csv('anomalies_iqr.csv')
anomalies_if.to_csv('anomalies_isolationforest.csv')


# 1. Load dataset
# -----------------------------
# Replace 'supply_chain_data.csv' with your actual file
df = pd.read_csv('supply_chain_cleaned_data.csv')